In [0]:
%run ./vector_search_setup

In [0]:
# Plain LLM vs RAG 

TEST_QUESTIONS = [
    "How does p-adic arithmetic coding generalize Huffman's algorithm and standard arithmetic coding?",
    "What approach has been proposed for using Kekulé states of polycyclic hydrocarbons as a basis for molecular computation?",
    "How can particle swarm optimization be used for designing fractional order PID controllers?",
    "What is gradient descent and how is it used in machine learning?",
]

all_results = []
for q in TEST_QUESTIONS:
    result = compare(q)
    all_results.append({"question": q, **result})
    print()

import json
from datetime import datetime

rows = []
for r in all_results:
    rows.append({
        "question": r["question"],
        "plain_answer": r["plain"]["answer"],
        "rag_answer": r["rag"]["answer"],
        "rag_sources": json.dumps(r["rag"]["sources"]),
        "plain_tokens": r["plain"]["tokens"],
        "plain_time_sec": r["plain"]["time_sec"],
        "rag_time_sec": r["rag"]["total_time_sec"],
        "tested_at": datetime.now().isoformat(),
    })

df_results = spark.createDataFrame(rows)
df_results.write.mode("overwrite").saveAsTable("arxivist_ai_dev.gold.rag_test_results")
print(f"\nResults saved to arxivist_ai_dev.gold.rag_test_results")
display(df_results)

In [0]:
dbutils.widgets.text("question", "What methods exist for non-negative matrix factorization with convexity constraints?", "Ask ArxiVist AI")

question = dbutils.widgets.get("question").strip()

if not question:
    print("Enter a question in the widget above and re-run this cell.")
else:
    print(f"\n{'='*80}")
    print(f"YOUR QUESTION: {question}")
    print(f"{'='*80}")

    # Plain LLM 
    print(f"\n\U0001f916 PLAIN LLM (no context):")
    print(f"{'-'*40}")
    plain = ask_plain(question)
    print(plain["answer"])
    print(f"\u23f1 {plain['time_sec']}s | Tokens: {plain['tokens']}")

    # RAG 
    print(f"\n\U0001f4da RAG (Vector Search + LLM):")
    print(f"{'-'*40}")
    rag = ask_rag(question)
    print(rag["answer"])
    print(f"\n\u23f1 {rag['total_time_sec']}s (retrieval: {rag['retrieval_time_sec']}s)")
    print(f"\nSources:")
    for title, score in rag["sources"]:
        print(f"  \u2022 [{score}] {title}")